In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, trim, lower
import logging
from typing import Optional

# Configuração de logging (continua)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- DEFINIÇÕES DE CAMINHOS ---
BASE_PATH = './data_lake' # Certifique-se que o BASE_PATH seja o mesmo!
SILVER_PATH = os.path.join(BASE_PATH, "silver")
GOLD_PATH = os.path.join(BASE_PATH, "gold", "breweries_by_type_and_location")
# ------------------------------

def create_spark_session() -> SparkSession:
    """Inicia a sessão Spark (reutilizada)."""
    spark = (
        SparkSession.builder
        .appName("BEES_PySpark_LoadGold")
        .master("local[*]")
        .getOrCreate()
    )
    logging.info("Spark session criada com sucesso.")
    return spark

def read_silver_data(spark: SparkSession, silver_path: str):
    """Lê os arquivos Parquet da camada Silver (particionados)."""
    logging.info(f"Lendo arquivos Parquet da camada Silver: {silver_path}")
    df = spark.read.parquet(silver_path)
    total_records = df.count()
    logging.info(f"Total de registros lidos da Silver: {total_records}")
    df.printSchema()
    return df

def aggregate_to_gold(df):
    """
    CRIA A VISÃO AGREGADA SOLICITADA (Item 5c):
    - Agrega a quantidade (count) de cervejarias.
    - Pelos campos: brewery_type, country e state_province.
    """
    logging.info("Executando agregação para Gold: COUNT de cervejarias por Type e Location.")
    
    # As colunas já foram limpas e padronizadas na Silver Layer
    df_gold = df.groupBy(col("brewery_type"), col("country"), col("state_province")) \
                  .agg(count("*").alias("brewery_count"))
    
    return df_gold

def write_gold_data(df, gold_path: str):
    """Grava o DataFrame Gold no formato Parquet (pode ser Delta ou outro)."""
    os.makedirs(os.path.dirname(gold_path), exist_ok=True)
    logging.info(f"Gravando camada Gold (Agregada) em: {gold_path}")
    # É uma agregação, não precisa de particionamento complexo, mas pode ser útil para dashboards
    df.write.mode("overwrite").parquet(gold_path)
    logging.info("Camada Gold gerada com sucesso.")
    
def run_gold_pipeline():
    os.environ['HADOOP_CONF_DIR'] = '' # Evita problemas comuns em Jupyter/local
    os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

    spark: Optional[SparkSession] = None
    try:
        spark = create_spark_session()

        # 1. Leitura Silver
        df_silver = read_silver_data(spark, SILVER_PATH)

        # 2. Agregação para Gold
        df_gold = aggregate_to_gold(df_silver)
        
        logging.info("Schema da Gold Layer (Agregada):")
        df_gold.printSchema()
        
        logging.info("Amostra da Gold Layer (Agregada):")
        df_gold.show(10, truncate=False)

        # 3. Escrita Gold
        write_gold_data(df_gold, GOLD_PATH)

        logging.info("Pipeline Silver → Gold concluído com sucesso!")

    except Exception as e:
        logging.error(f"Erro ao gerar Gold: {e}")
        import traceback
        logging.error(traceback.format_exc())
    finally:
        if spark:
            logging.info("Encerrando Spark session...")
            spark.stop()


2025-10-21 00:28:17,325 - INFO - Spark session criada com sucesso.
2025-10-21 00:28:17,329 - INFO - Lendo arquivos Parquet da camada Silver: ./data_lake/silver
2025-10-21 00:28:39,573 - INFO - Total de registros lidos da Silver: 8923
2025-10-21 00:28:39,581 - INFO - Executando agregação para Gold: COUNT de cervejarias por Type e Location.
2025-10-21 00:28:39,696 - INFO - Schema da Gold Layer (Agregada):
2025-10-21 00:28:39,698 - INFO - Amostra da Gold Layer (Agregada):


root
 |-- address_1: string (nullable = true)
 |-- address_2: string (nullable = true)
 |-- address_3: string (nullable = true)
 |-- brewery_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- name: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- state: string (nullable = true)
 |-- street: string (nullable = true)
 |-- website_url: string (nullable = true)
 |-- load_timestamp: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state_province: string (nullable = true)

root
 |-- brewery_type: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state_province: string (nullable = true)
 |-- brewery_count: long (nullable = false)



2025-10-21 00:28:50,736 - INFO - Gravando camada Gold (Agregada) em: ./data_lake/gold/breweries_by_type_and_location


+------------+-------------+--------------+-------------+
|brewery_type|country      |state_province|brewery_count|
+------------+-------------+--------------+-------------+
|micro       |united states|oregon        |142          |
|micro       |united states|missouri      |67           |
|large       |united states|missouri      |1            |
|micro       |united states|ohio          |163          |
|proprietor  |united states|indiana       |1            |
|planning    |united states|illinois      |23           |
|contract    |united states|georgia       |2            |
|micro       |united states|pennsylvania  |129          |
|contract    |united states|florida       |6            |
|regional    |united states|new hampshire |1            |
+------------+-------------+--------------+-------------+
only showing top 10 rows



2025-10-21 00:29:00,138 - INFO - Camada Gold gerada com sucesso.
2025-10-21 00:29:00,139 - INFO - Pipeline Silver → Gold concluído com sucesso!
2025-10-21 00:29:00,141 - INFO - Encerrando Spark session...


In [ ]:

run_gold_pipeline()